In [1]:
from datasets import load_dataset, Audio
from transformers import AutoFeatureExtractor
import numpy as np

In [2]:
ds = load_dataset("sanchit-gandhi/gtzan")
ds = ds["train"].train_test_split(test_size=0.1, shuffle=True)
id2label_fn = ds["train"].features["genre"].int2str

In [ ]:
import gradio as gr


def generate_audio():
    example = ds["train"].shuffle()[0]
    audio = example["audio"]
    return (
        audio["sampling_rate"],
        audio["array"],
    ), id2label_fn(example["genre"])


with gr.Blocks() as demo:
    with gr.Column():
        for _ in range(4):
            audio, label = generate_audio()
            output = gr.Audio(audio, label=label)

demo.launch(debug=True)

In [4]:
model_id = "MIT/ast-finetuned-audioset-10-10-0.4593"

feature_extractor = AutoFeatureExtractor.from_pretrained(model_id, do_normalize=True, return_attention_mask=True)
feature_extractor

ASTFeatureExtractor {
  "do_normalize": true,
  "feature_extractor_type": "ASTFeatureExtractor",
  "feature_size": 1,
  "max_length": 1024,
  "mean": -4.2677393,
  "num_mel_bins": 128,
  "padding_side": "right",
  "padding_value": 0.0,
  "return_attention_mask": true,
  "sampling_rate": 16000,
  "std": 4.5689974
}

In [5]:
ds = ds.cast_column("audio", Audio(sampling_rate=feature_extractor.sampling_rate))

In [7]:
sample = ds['train'][0]["audio"]
print(f"Mean: {np.mean(sample['array']):.3}, Variance: {np.var(sample['array']):.3}")

Mean: -8.49e-06, Variance: 0.0306


In [11]:
inputs = feature_extractor(sample["array"], sampling_rate=sample["sampling_rate"], do_normalize=True, return_attention_mask=True)

print(f"inputs keys: {list(inputs.keys())}")

print(
    f"Mean: {np.mean(inputs['input_values']):.3}, Variance: {np.var(inputs['input_values']):.3}"
)

inputs keys: ['input_values']
Mean: 0.316, Variance: 0.0952


In [14]:
def preprocess_audio(examples):
    audio_arrays = [x['array'] for x in examples["audio"]]
    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=feature_extractor.sampling_rate,
        max_length=int(feature_extractor.sampling_rate * 30.0),
        truncation=True,
        return_attention_mask=True,
    )
    return inputs

In [15]:
ds_encoded = ds.map(
    preprocess_audio,
    remove_columns=["audio", "file"],
    batched=True,
    batch_size=100,
    num_proc=1
)

Map (num_proc=1):   0%|          | 0/899 [00:00<?, ? examples/s]

TimeoutError: 